In [ ]:
PROJ_NAME = "dilithium_sign_ttest"
LABEL = ""
OFFSET_STEP = 120000
TOTAL_SAMPLES = 1229144

In [ ]:
import sys
sys.path.insert(0, "../")
from host.ttest_trace_collector import TTestTraceCollector

In [ ]:
trace_collector = TTestTraceCollector(proj_name=PROJ_NAME, label=LABEL, input_len=32,
                                      output_len=0, offset=0)

In [ ]:
trace_collector.init_target(force=False)

In [ ]:
trace_collector.default_setup(40E6, 12, OFFSET_STEP)

In [ ]:
trace_collector.build_fw()
trace_collector.program_fw()

# PRNG=OFF

In [ ]:
from tqdm import tqdm

N=1000

for offset in tqdm(range(0, TOTAL_SAMPLES, OFFSET_STEP)):
    trace_collector.set_offset(offset)
    trace_collector.collect_traces(N=N, prng_off=True, overwrite=True, check_output=False, random_order=False, coin_flip=True)
    ttest_analysis_off = trace_collector.get_analysis_obj(N=N, prng_off=True)
    ttest_analysis_off.run_ttest()
    ttest_analysis_off.compute_means()
    ttest_analysis_off.save_ttest_results()
    trace_collector.delete_traces(N=N, prng_off=True)

# PRNG=ON

In [ ]:
import os

TRASH_PATH = os.path.expanduser("~/.local/share/Trash")

def empty_trash():
    for sub in ("files", "info"):
        path = os.path.join(TRASH_PATH, sub)
        if not os.path.exists(path):
            continue

        for item in os.listdir(path):
            item_path = os.path.join(path, item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.remove(item_path)
                elif os.path.isdir(item_path):
                    os.rmdir(item_path)  # only works if empty
            except Exception as e:
                print(f"Failed to delete {item_path}: {e}")

In [ ]:
from tqdm import tqdm

N = 100000

print("ab seed is ", trace_collector.ab_seq_seed)
print("ttest seed is ", trace_collector.const_seed)

for offset in tqdm(range(0, TOTAL_SAMPLES, OFFSET_STEP)):
    trace_collector.set_offset(offset)
    trace_collector.collect_traces(N=10, prng_off=False, overwrite=True, check_output=False, random_order=False, coin_flip=True)
    trace_collector.delete_traces(N=10, prng_off=False)
    trace_collector.collect_traces(N=N, prng_off=False, overwrite=True, check_output=False, random_order=False, coin_flip=True)
    ttest_analysis_on = trace_collector.get_analysis_obj(N=N, prng_off=False)
    ttest_analysis_on.run_ttest(chunk=5000, force_equal=False)
    ttest_analysis_on.compute_means(num_traces=100)
    ttest_analysis_on.save_ttest_results()
    trace_collector.delete_traces(N=N, prng_off=False)